# Inference Latency Benchmark

This notebook benchmarks the inference-time computational overhead of the
generation methods evaluated in the thesis.

All measurements are performed using single-prompt generation (`batch_size=1`)
on the same deterministic 12-condition benchmark containing two examples from
each CEFR level (A1–C2).

The benchmark reports:

- Time to First Token (TTFT);
- Time to Last Token (TTLT);
- overall generation throughput;
- autoregressive decode throughput.

The benchmark is executed on an NVIDIA L4 GPU using the same generation
configuration wherever applicable.

## Part I — Base and Optimization-Based Control

Benchmarks:

- Base LLM (Prompt-Only)
- PPLM

In [ ]:
!pip install -q transformers torch datasets tqdm huggingface_hub accelerate scikit-learn textstat spacy hf_transfer
!python -m spacy download en_core_web_sm

import os
import gc
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login, hf_hub_download
from google.colab import drive


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 113.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 150.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# ==========================================
# 0. CONFIGURATION & CLOUD DIRECTORIES
# ==========================================
drive.mount('/content/drive')

INPUT_CSV_PATH = "/content/drive/MyDrive/Your_Path/"
    "in_domain_evaluation_prompt_matrix.csv"
LATENCY_DIR = "/content/drive/MyDrive/Your_Path/"
    "inference_latency_benchmark"
os.makedirs(LATENCY_DIR, exist_ok=True)

SUBSET_12_CSV = os.path.join(LATENCY_DIR, "latency_benchmark_subset_12.csv")
BASE_CSV_PATH = os.path.join(LATENCY_DIR, "base_llm_latency.csv")
PPLM_CSV_PATH = os.path.join(LATENCY_DIR, "pplm_latency.csv")
PMT_CSV_PATH = os.path.join(LATENCY_DIR, "pmt_latency.csv")
SUMMARY_TXT_PATH = os.path.join(LATENCY_DIR, "latency_summary_report.txt")

hf_token = "HF_Token"
login(token=hf_token)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
inv_label_map = {v: k for k, v in label_map.items()}


Mounted at /content/drive
Using device: cuda


## Latency Benchmark Subset

A deterministic subset of 12 conditions is sampled from the final
In-Domain Evaluation Prompt Matrix:

- 2 conditions per CEFR level;
- 6 CEFR levels;
- 12 conditions total;
- sampling seed: 42.

The same subset is reused for every compared method.

In [ ]:
# ==========================================
# 1. EXTRACT & SAVE 12-PROMPT REPRODUCIBLE SUBSET
# ==========================================
df_master = pd.read_csv(INPUT_CSV_PATH)
df_master['cefr'] = df_master['cefr'].astype(str).str.strip().str.upper()

if os.path.exists(SUBSET_12_CSV):
    print(f"Loading existing 12-prompt benchmark subset from: {SUBSET_12_CSV}")
    df_subset = pd.read_csv(SUBSET_12_CSV)
else:
    print("Extracting deterministic 12-prompt subset (2 prompts per CEFR level A1-C2)...")
    np.random.seed(42)
    selected_rows = []
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        level_df = df_master[df_master['cefr'] == level]
        chosen = level_df.sample(n=2, random_state=42)
        selected_rows.append(chosen)

    df_subset = pd.concat(selected_rows, ignore_index=True)
    df_subset.to_csv(SUBSET_12_CSV, index=False)
    print(f"Saved 12-prompt benchmark subset to: {SUBSET_12_CSV}")

print(f"Benchmark Subset Verification: {len(df_subset)} prompts loaded.")

Loading existing 12-prompt benchmark subset from: /content/drive/MyDrive/Mohammd_Thesis/Results/Latency/latency_benchmark_subset_12.csv
Benchmark Subset Verification: 12 prompts loaded.


In [ ]:
# ==========================================
# 2. LOAD LLM & TOKENIZER
# ==========================================
print("\nLoading Llama-3.1-8B-Instruct Base Model...")
model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

eos_ids = [tokenizer.eos_token_id]
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if eot_id is not None:
    eos_ids.append(eot_id)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)
base_model.eval()

# Sampling Hyperparameters
TEMPERATURE = 0.6
REPETITION_PENALTY = 1.15
MAX_NEW_TOKENS = 200
TOP_K = 50

def build_formatted_prompt(topic_title, t_cefr):
    sys_instr = (
        f"You are an expert English language teacher demonstrating CEFR proficiency levels. "
        f"Your task is to write a flawless, grammatically correct text responding to this prompt: '{topic_title}'. "
        f"The output must serve as a perfect textbook example of strictly {t_cefr} level English. "
        f"If the requested target level is A1/A2, use very simple vocabulary, short sentences, and primitive structures. "
        f"If the requested target level is C1/C2, utilize highly advanced vocabulary, idioms, and complex sentence patterns. "
        f"Write only the direct response. Do not write any meta-commentary, greetings, or conversational pleasantries."
    )
    messages = [{"role": "system", "content": sys_instr}, {"role": "user", "content": topic_title}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


Loading Llama-3.1-8B-Instruct Base Model...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
# ==========================================
# 3. BENCHMARK METHOD 1: BASE LLM
# ==========================================
print("\n" + "="*60)
print("🚀 BENCHMARKING METHOD 1: BASE LLM (UNSTEERED)")
print("="*60)

# Warm-up pass to eliminate GPU cold-start skew
warmup_prompt = build_formatted_prompt("Warmup Prompt", "B1")
warmup_inputs = tokenizer(warmup_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    _ = base_model.generate(**warmup_inputs, max_new_tokens=10, do_sample=False)
torch.cuda.synchronize()

base_latency_records = []

for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Base LLM Latency"):
    t_cefr = str(row['cefr']).strip().upper()
    formatted_prompt = build_formatted_prompt(row['topic_title'], t_cefr)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask

    generated_tokens = []
    past_key_values = None
    current_attention_mask = attention_mask

    torch.cuda.synchronize()
    start_time = time.perf_counter()
    ttft = None

    for step in range(MAX_NEW_TOKENS):
        # FIX: Directly use next_token_id (shape is already [1, 1]) without unsqueeze
        model_input = input_ids if step == 0 else next_token_id

        if step > 0:
            current_attention_mask = torch.cat([current_attention_mask, torch.ones(1, 1, dtype=torch.long, device=device)], dim=-1)

        with torch.no_grad():
            outputs = base_model(
                input_ids=model_input,
                attention_mask=current_attention_mask,
                past_key_values=past_key_values,
                use_cache=True
            )
        past_key_values = outputs.past_key_values
        lm_logits = outputs.logits[:, -1, :]

        # Repetition penalty & sampling
        for tok_id in set(generated_tokens + input_ids[0].tolist()):
            if lm_logits[0, tok_id] > 0: lm_logits[0, tok_id] /= REPETITION_PENALTY
            else: lm_logits[0, tok_id] *= REPETITION_PENALTY

        lm_logits = lm_logits / TEMPERATURE
        values, indices = torch.topk(lm_logits, TOP_K, dim=-1)
        filtered_logits = torch.full_like(lm_logits, float('-inf')).scatter_(1, indices, values)
        probs_lm = F.softmax(filtered_logits, dim=-1)

        # This returns shape [1, 1]
        next_token_id = torch.multinomial(probs_lm, num_samples=1)

        if step == 0:
            torch.cuda.synchronize()
            ttft = time.perf_counter() - start_time

        tok_item = next_token_id.item()
        generated_tokens.append(tok_item)
        if tok_item in eos_ids:
            break

    torch.cuda.synchronize()
    ttlt = time.perf_counter() - start_time

    num_tokens = len(generated_tokens)
    gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    gen_speed = num_tokens / ttlt if ttlt > 0 else 0.0
    decode_speed = (num_tokens - 1) / (ttlt - ttft) if (ttlt - ttft) > 0 and num_tokens > 1 else gen_speed

    base_latency_records.append({
        "topic_id": row['topic_id'],
        "topic_title": row['topic_title'],
        "target_cefr": t_cefr,
        "llm_output": gen_text,
        "num_generated_tokens": num_tokens,
        "ttft_seconds": round(ttft, 4),
        "ttlt_seconds": round(ttlt, 4),
        "gen_speed_tokens_per_sec": round(gen_speed, 2),
        "decode_speed_tokens_per_sec": round(decode_speed, 2)
    })

df_base_latency = pd.DataFrame(base_latency_records)
df_base_latency.to_csv(BASE_CSV_PATH, index=False)
print(f"✔️ Base LLM latency results saved to: {BASE_CSV_PATH}")


🚀 BENCHMARKING METHOD 1: BASE LLM (UNSTEERED)


Base LLM Latency: 100%|██████████| 12/12 [02:52<00:00, 14.36s/it]

✔️ Base LLM latency results saved to: /content/drive/MyDrive/Mohammd_Thesis/Results/Latency/base_llm_latency.csv


In [ ]:
# ==========================================
# 4. BENCHMARK METHOD 2: PPLM (WINNER: STEP 0.5, GRADS 20)
# ==========================================
print("\n" + "="*60)
print("🚀 BENCHMARKING METHOD 2: PURE PPLM WINNER (STEP 0.5 | GRADS 20)")
print("="*60)

class CEFR2LayerMLPHead(nn.Module):
    def __init__(self, input_dim=4096, hidden_dim=512, num_classes=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Dropout(p=0.35), nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.ReLU(),
            nn.Dropout(p=0.25), nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x): return self.network(x)

target_repo_id = "MohammadKhosravi/llama3.1-8b-cefr-steering-2layer-head-ordinal-universal"
steering_head = CEFR2LayerMLPHead().to(device)
weights_path = hf_hub_download(repo_id=target_repo_id, filename="cefr_steering_head.pt")
steering_head.load_state_dict(torch.load(weights_path, map_location=device))
steering_head.eval()

criterion = nn.CrossEntropyLoss(reduction='none')
STEP_SIZE = 0.5
GRAD_STEPS = 20

# Warm-up pass for PPLM
warmup_inputs = tokenizer(warmup_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    _ = base_model(input_ids=warmup_inputs.input_ids, use_cache=True, output_hidden_states=True)
torch.cuda.synchronize()

pplm_latency_records = []

for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="PPLM Latency"):
    t_cefr = str(row['cefr']).strip().upper()
    formatted_prompt = build_formatted_prompt(row['topic_title'], t_cefr)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask
    target_tensor = torch.tensor([label_map[t_cefr]], device=device)

    generated_tokens = []
    past_key_values = None
    current_attention_mask = attention_mask

    torch.cuda.synchronize()
    start_time = time.perf_counter()
    ttft = None

    for step in range(MAX_NEW_TOKENS):
        # FIX: Directly use next_token_id without unsqueeze
        model_input = input_ids if step == 0 else next_token_id

        if step > 0:
            current_attention_mask = torch.cat([current_attention_mask, torch.ones(1, 1, dtype=torch.long, device=device)], dim=-1)

        with torch.no_grad():
            outputs = base_model(
                input_ids=model_input,
                attention_mask=current_attention_mask,
                past_key_values=past_key_values,
                use_cache=True,
                output_hidden_states=True
            )
        past_key_values = outputs.past_key_values

        hidden_state_prenorm = outputs.hidden_states[-2][:, -1, :].detach().clone().to(torch.float32)
        hidden_state_prenorm.requires_grad_(True)

        # Autoregressive Gradient Ascent (20 steps)
        for g_step in range(GRAD_STEPS):
            class_logits = steering_head(hidden_state_prenorm)
            loss = criterion(class_logits, target_tensor).sum()
            if loss.item() == 0: break
            loss.backward()

            with torch.no_grad():
                raw_grad = hidden_state_prenorm.grad.data
                grad_norms = torch.norm(raw_grad, p=2, dim=-1, keepdim=True) + 1e-9
                normalized_grad = raw_grad / grad_norms
                hidden_state_prenorm.data = hidden_state_prenorm.data - (STEP_SIZE * normalized_grad)
                hidden_state_prenorm.grad.zero_()

        # Project modified hidden state to vocabulary logits
        with torch.no_grad():
            steered_normed = base_model.model.norm(hidden_state_prenorm.to(torch.bfloat16))
            lm_logits = base_model.lm_head(steered_normed)

            for tok_id in set(generated_tokens + input_ids[0].tolist()):
                if lm_logits[0, tok_id] > 0: lm_logits[0, tok_id] /= REPETITION_PENALTY
                else: lm_logits[0, tok_id] *= REPETITION_PENALTY

            lm_logits = lm_logits / TEMPERATURE
            values, indices = torch.topk(lm_logits, TOP_K, dim=-1)
            filtered_logits = torch.full_like(lm_logits, float('-inf')).scatter_(1, indices, values)
            probs_lm = F.softmax(filtered_logits, dim=-1)

            # This returns shape [1, 1]
            next_token_id = torch.multinomial(probs_lm, num_samples=1)

        if step == 0:
            torch.cuda.synchronize()
            ttft = time.perf_counter() - start_time

        tok_item = next_token_id.item()
        generated_tokens.append(tok_item)
        if tok_item in eos_ids:
            break

    torch.cuda.synchronize()
    ttlt = time.perf_counter() - start_time

    num_tokens = len(generated_tokens)
    gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    gen_speed = num_tokens / ttlt if ttlt > 0 else 0.0
    decode_speed = (num_tokens - 1) / (ttlt - ttft) if (ttlt - ttft) > 0 and num_tokens > 1 else gen_speed

    pplm_latency_records.append({
        "topic_id": row['topic_id'],
        "topic_title": row['topic_title'],
        "target_cefr": t_cefr,
        "llm_output": gen_text,
        "num_generated_tokens": num_tokens,
        "ttft_seconds": round(ttft, 4),
        "ttlt_seconds": round(ttlt, 4),
        "gen_speed_tokens_per_sec": round(gen_speed, 2),
        "decode_speed_tokens_per_sec": round(decode_speed, 2)
    })

df_pplm_latency = pd.DataFrame(pplm_latency_records)
df_pplm_latency.to_csv(PPLM_CSV_PATH, index=False)
print(f"✔️ PPLM latency results saved to: {PPLM_CSV_PATH}")

del steering_head
torch.cuda.empty_cache()
gc.collect()


🚀 BENCHMARKING METHOD 2: PURE PPLM WINNER (STEP 0.5 | GRADS 20)


cefr_steering_head.pt: reconstructing file:   0%|          |  0.00B / 8.41MB            

cefr_steering_head.pt: downloading bytes:           |  0.00B            

PPLM Latency: 100%|██████████| 12/12 [03:38<00:00, 18.21s/it]


✔️ PPLM latency results saved to: /content/drive/MyDrive/Mohammd_Thesis/Results/Latency/pplm_latency.csv


206

In [ ]:
# ==========================================
# 5. BENCHMARK METHOD 3: PREFIXMEMORY-TUNING (PMT)
# ==========================================
print("\n" + "="*60)
print("🚀 BENCHMARKING METHOD 3: PREFIXMEMORY-TUNING (PMT)")
print("="*60)

MEMORY_RANK = 64

class PrefixMemoryController(nn.Module):
    def __init__(self, num_layers=32, hidden_dim=4096, num_classes=6, embed_dim=128, rank=64):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.rank = rank

        self.alpha = nn.Parameter(torch.tensor(0.1))
        self.cefr_embedding = nn.Embedding(num_classes, embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_layers * rank)
        )

        self.W_down = nn.Parameter(torch.randn(num_layers, hidden_dim, rank) / (hidden_dim**0.5))
        w_up_tensor = torch.empty(num_layers, rank, hidden_dim)
        nn.init.xavier_uniform_(w_up_tensor)
        self.W_up = nn.Parameter(w_up_tensor)

    def forward(self, cefr_ids):
        embs = self.cefr_embedding(cefr_ids)
        modulation = self.mlp(embs).view(-1, self.num_layers, self.rank)
        return modulation

prefix_memory_controller = PrefixMemoryController(rank=MEMORY_RANK).to(device).to(torch.bfloat16)

pmt_weights_path = hf_hub_download(
    repo_id="MohammadKhosravi/llama3.1-8b-prefixmemory-cefr-balanced-topic-aligned",
    filename="prefixmemory_controller_weights.pt"
)
prefix_memory_controller.load_state_dict(torch.load(pmt_weights_path, map_location=device))
prefix_memory_controller.eval()

current_single_cefr_modulation = None

def make_pmt_single_hook(layer_idx):
    def hook_fn(module, args, kwargs, output):
        global current_single_cefr_modulation
        if current_single_cefr_modulation is None: return output

        if len(args) > 0: hidden_states = args[0]
        elif 'hidden_states' in kwargs: hidden_states = kwargs['hidden_states']
        else: return output

        attn_output = output[0]

        W_d = prefix_memory_controller.W_down[layer_idx]
        W_u = prefix_memory_controller.W_up[layer_idx]

        query_states = torch.matmul(hidden_states, W_d)
        mod = current_single_cefr_modulation[:, layer_idx, :].unsqueeze(1)
        phi_X = F.elu(query_states * mod)
        prefix_memory_bias = torch.matmul(phi_X, W_u)

        final_attn_output = attn_output + (prefix_memory_controller.alpha * prefix_memory_bias)
        return (final_attn_output,) + output[1:]
    return hook_fn

# Register PMT hooks across all 32 layers
for i in range(32):
    base_model.model.layers[i].self_attn.register_forward_hook(make_pmt_single_hook(i), with_kwargs=True)

# Warm-up pass for PMT
warmup_cefr = torch.tensor([label_map["B1"]], device=device)
current_single_cefr_modulation = prefix_memory_controller(warmup_cefr)
with torch.no_grad():
    _ = base_model(input_ids=warmup_inputs.input_ids, use_cache=True)
torch.cuda.synchronize()

pmt_latency_records = []

for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="PMT Latency"):
    t_cefr = str(row['cefr']).strip().upper()
    formatted_prompt = build_formatted_prompt(row['topic_title'], t_cefr)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask

    cefr_tensor = torch.tensor([label_map[t_cefr]], device=device)
    current_single_cefr_modulation = prefix_memory_controller(cefr_tensor)

    generated_tokens = []
    past_key_values = None
    current_attention_mask = attention_mask

    torch.cuda.synchronize()
    start_time = time.perf_counter()
    ttft = None

    for step in range(MAX_NEW_TOKENS):
        # FIX: Directly use next_token_id without unsqueeze
        model_input = input_ids if step == 0 else next_token_id

        if step > 0:
            current_attention_mask = torch.cat([current_attention_mask, torch.ones(1, 1, dtype=torch.long, device=device)], dim=-1)

        with torch.no_grad():
            outputs = base_model(
                input_ids=model_input,
                attention_mask=current_attention_mask,
                past_key_values=past_key_values,
                use_cache=True
            )
        past_key_values = outputs.past_key_values
        lm_logits = outputs.logits[:, -1, :]

        for tok_id in set(generated_tokens + input_ids[0].tolist()):
            if lm_logits[0, tok_id] > 0: lm_logits[0, tok_id] /= REPETITION_PENALTY
            else: lm_logits[0, tok_id] *= REPETITION_PENALTY

        lm_logits = lm_logits / TEMPERATURE
        values, indices = torch.topk(lm_logits, TOP_K, dim=-1)
        filtered_logits = torch.full_like(lm_logits, float('-inf')).scatter_(1, indices, values)
        probs_lm = F.softmax(filtered_logits, dim=-1)

        # This returns shape [1, 1]
        next_token_id = torch.multinomial(probs_lm, num_samples=1)

        if step == 0:
            torch.cuda.synchronize()
            ttft = time.perf_counter() - start_time

        tok_item = next_token_id.item()
        generated_tokens.append(tok_item)
        if tok_item in eos_ids:
            break

    torch.cuda.synchronize()
    ttlt = time.perf_counter() - start_time

    num_tokens = len(generated_tokens)
    gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    gen_speed = num_tokens / ttlt if ttlt > 0 else 0.0
    decode_speed = (num_tokens - 1) / (ttlt - ttft) if (ttlt - ttft) > 0 and num_tokens > 1 else gen_speed

    pmt_latency_records.append({
        "topic_id": row['topic_id'],
        "topic_title": row['topic_title'],
        "target_cefr": t_cefr,
        "llm_output": gen_text,
        "num_generated_tokens": num_tokens,
        "ttft_seconds": round(ttft, 4),
        "ttlt_seconds": round(ttlt, 4),
        "gen_speed_tokens_per_sec": round(gen_speed, 2),
        "decode_speed_tokens_per_sec": round(decode_speed, 2)
    })

df_pmt_latency = pd.DataFrame(pmt_latency_records)
df_pmt_latency.to_csv(PMT_CSV_PATH, index=False)
print(f"✔️ PMT latency results saved to: {PMT_CSV_PATH}")


🚀 BENCHMARKING METHOD 3: PREFIXMEMORY-TUNING (PMT)


prefixmemory_controller_weights.pt: reconstructing file:   0%|          |  0.00B / 34.7MB            

prefixmemory_controller_weights.pt: downloading bytes:           |  0.00B            

PMT Latency: 100%|██████████| 12/12 [01:58<00:00,  9.86s/it]

✔️ PMT latency results saved to: /content/drive/MyDrive/Mohammd_Thesis/Results/Latency/pmt_latency.csv


In [ ]:
# ==========================================
# 6. COMPILE & WRITE SUMMARY REPORT
# ==========================================
base_avg_ttft = df_base_latency['ttft_seconds'].mean()
base_avg_ttlt = df_base_latency['ttlt_seconds'].mean()
base_avg_speed = df_base_latency['gen_speed_tokens_per_sec'].mean()
base_avg_decode = df_base_latency['decode_speed_tokens_per_sec'].mean()

pplm_avg_ttft = df_pplm_latency['ttft_seconds'].mean()
pplm_avg_ttlt = df_pplm_latency['ttlt_seconds'].mean()
pplm_avg_speed = df_pplm_latency['gen_speed_tokens_per_sec'].mean()
pplm_avg_decode = df_pplm_latency['decode_speed_tokens_per_sec'].mean()

pmt_avg_ttft = df_pmt_latency['ttft_seconds'].mean()
pmt_avg_ttlt = df_pmt_latency['ttlt_seconds'].mean()
pmt_avg_speed = df_pmt_latency['gen_speed_tokens_per_sec'].mean()
pmt_avg_decode = df_pmt_latency['decode_speed_tokens_per_sec'].mean()

summary_text = "="*80 + "\n"
summary_text += " ⏱️ THESIS INFERENCE LATENCY BENCHMARK REPORT (12-PROMPT SINGLE GENERATION)\n"
summary_text += "="*80 + "\n"
summary_text += f"Benchmark Location : {LATENCY_DIR}\n"
summary_text += f"Hardware Engine    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\n"
summary_text += f"Evaluation Mode    : Single Prompt Generation (Batch Size = 1)\n"
summary_text += "="*80 + "\n\n"

summary_text += f"{'Method':<28} | {'Avg TTFT (s)':<12} | {'Avg TTLT (s)':<12} | {'Gen Speed (tok/s)':<18} | {'Latency Overhead':<16}\n"
summary_text += "-"*95 + "\n"

summary_text += f"{'Base LLM (Unsteered)':<28} | {base_avg_ttft:<12.4f} | {base_avg_ttlt:<12.4f} | {base_avg_speed:<18.2f} | 1.00x (Baseline)\n"
summary_text += f"{'PPLM Winner (Step 0.5/G20)':<28} | {pplm_avg_ttft:<12.4f} | {pplm_avg_ttlt:<12.4f} | {pplm_avg_speed:<18.2f} | {pplm_avg_ttlt/base_avg_ttlt:.2f}x Slowdown\n"
summary_text += f"{'PrefixMemory-Tuning (PMT)':<28} | {pmt_avg_ttft:<12.4f} | {pmt_avg_ttlt:<12.4f} | {pmt_avg_speed:<18.2f} | {pmt_avg_ttlt/base_avg_ttlt:.2f}x Slowdown\n"
summary_text += "-"*95 + "\n\n"

summary_text += "📊 DETAILED LATENCY METRIC BREAKDOWN:\n"
summary_text += f"• Base LLM  -> TTFT: {base_avg_ttft:.4f}s | TTLT: {base_avg_ttlt:.4f}s | Decode Speed: {base_avg_decode:.2f} tokens/s\n"
summary_text += f"• PPLM      -> TTFT: {pplm_avg_ttft:.4f}s | TTLT: {pplm_avg_ttlt:.4f}s | Decode Speed: {pplm_avg_decode:.2f} tokens/s\n"
summary_text += f"• PMT       -> TTFT: {pmt_avg_ttft:.4f}s | TTLT: {pmt_avg_ttlt:.4f}s | Decode Speed: {pmt_avg_decode:.2f} tokens/s\n"

with open(SUMMARY_TXT_PATH, "w") as f:
    f.write(summary_text)

print("\n" + summary_text)
print(f"\n🎉 LATENCY BENCHMARKING COMPLETE! All artifacts saved in: {LATENCY_DIR}")


 ⏱️ THESIS INFERENCE LATENCY BENCHMARK REPORT (12-PROMPT SINGLE GENERATION)
Benchmark Location : /content/drive/MyDrive/Mohammd_Thesis/Results/Latency
Hardware Engine    : NVIDIA L4
Evaluation Mode    : Single Prompt Generation (Batch Size = 1)

Method                       | Avg TTFT (s) | Avg TTLT (s) | Gen Speed (tok/s)  | Latency Overhead
-----------------------------------------------------------------------------------------------
Base LLM (Unsteered)         | 0.1151       | 14.3616      | 13.54              | 1.00x (Baseline)
PPLM Winner (Step 0.5/G20)   | 0.1232       | 18.2043      | 10.99              | 1.27x Slowdown
PrefixMemory-Tuning (PMT)    | 0.0990       | 9.8560       | 13.48              | 0.69x Slowdown
-----------------------------------------------------------------------------------------------

📊 DETAILED LATENCY METRIC BREAKDOWN:
• Base LLM  -> TTFT: 0.1151s | TTLT: 14.3616s | Decode Speed: 13.58 tokens/s
• PPLM      -> TTFT: 0.1232s | TTLT: 18.2043s | Decode

## Part II — Parameter-Efficient and Architectural Controllers

Benchmarks:

- LoRA
- Standard Prefix-Tuning
- CEFR Multi-Prefix Tuning (~537M)
- CEFR-Gated PMT

In [ ]:
# ============================================================
# THESIS INFERENCE LATENCY BENCHMARK
# ADDITIONAL FOUR REPRESENTATIVE MODELS
# ============================================================
# Models:
#   1. LoRA 6k
#   2. Standard Prefix-Tuning 6k
#   3. CEFR Multi-Prefix Tuning ~537M
#   4. Proposed CEFR-Gated PMT ~537M
# Benchmark: Existing deterministic 12-prompt benchmark (Batch size = 1)
# Metrics: TTFT, TTLT, Generation speed, Decode speed
# ============================================================

# ============================================================
# 0. INSTALL DEPENDENCIES & 1. IMPORTS
# ============================================================
!pip install -q transformers peft torch pandas tqdm huggingface_hub accelerate hf_transfer "torchao>=0.16.0"

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import gc
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.cache_utils import DynamicCache
from transformers.generation.streamers import BaseStreamer
from peft import PeftModel
from huggingface_hub import login, hf_hub_download, HfApi
from google.colab import drive, userdata

# ============================================================
# 2. REPRODUCIBILITY & 3. GOOGLE DRIVE
# ============================================================
set_seed(42)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

drive.mount("/content/drive")

# ============================================================
# 4. PATHS
# ============================================================
LATENCY_DIR = "/content/drive/MyDrive/Your_Path/"
    "inference_latency_benchmark"
os.makedirs(LATENCY_DIR, exist_ok=True)

SUBSET_12_CSV = os.path.join(LATENCY_DIR, "latency_benchmark_subset_12.csv")
LORA_CSV_PATH = os.path.join(LATENCY_DIR, "lora_latency.csv")
STANDARD_PT_CSV_PATH = os.path.join(LATENCY_DIR, "standard_prefix_tuning_latency.csv")
CEFR_PT_537M_CSV_PATH = os.path.join(LATENCY_DIR, "cefr_pt_537m_latency.csv")
CEFR_GATED_PMT_CSV_PATH = os.path.join(LATENCY_DIR, "cefr_gated_pmt_latency.csv")
SUMMARY_TXT_PATH = os.path.join(LATENCY_DIR, "latency_four_models_summary.txt")

# ============================================================
# 5. HUGGING FACE AUTHENTICATION & 6. DEVICE
# ============================================================
try:
    login(token=userdata.get("HF_TOKEN"))
    print("✔ Hugging Face authentication successful.")
except Exception as e:
    print("⚠ Could not read HF_TOKEN from Colab Secrets.\n", e)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using execution device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# 7. MODEL REPOSITORIES & 8. CEFR LABEL MAP & 9. CONFIG
# ============================================================
BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
LORA_REPO_ID = "MohammadKhosravi/llama3.1-8b-lora-cefr-steering-6k"
STANDARD_PT_REPO_ID = "MohammadKhosravi/llama3.1-8b-standard-prefix-tuning-6k"
CEFR_PT_537M_REPO_ID = "MohammadKhosravi/llama3.1-8b-cefr-pt-537m-param-matched"
CEFR_GATED_PMT_REPO_ID = "MohammadKhosravi/llama3.1-8b-pure-pmt-cefr-gating-6k"

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
TEMPERATURE, REPETITION_PENALTY, TOP_K, MAX_NEW_TOKENS = 0.6, 1.15, 50, 200

# ============================================================
# 10. LOAD EXISTING 12-PROMPT BENCHMARK
# ============================================================
if not os.path.exists(SUBSET_12_CSV):
    raise FileNotFoundError(f"12-prompt latency benchmark not found:\n{SUBSET_12_CSV}")

df_subset = pd.read_csv(SUBSET_12_CSV)
required_columns = {"topic_id", "topic_title", "cefr"}
missing_columns = required_columns - set(df_subset.columns)
if missing_columns:
    raise ValueError(f"Benchmark CSV missing columns: {missing_columns}")

df_subset["cefr"] = df_subset["cefr"].astype(str).str.strip().str.upper()
if len(df_subset) != 12:
    raise ValueError(f"Expected exactly 12 benchmark rows, found {len(df_subset)}.")

print("\n" + "=" * 70 + "\n12-PROMPT LATENCY BENCHMARK\n" + "=" * 70)
print(f"Dataset: {SUBSET_12_CSV}\nRows: {len(df_subset)}")
print(df_subset["cefr"].value_counts().sort_index())
print("=" * 70)

# ============================================================
# 11. TOKENIZER & 12. EOS IDS
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

eos_ids = [tokenizer.eos_token_id]
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if eot_id is not None and eot_id >= 0 and eot_id not in eos_ids:
    eos_ids.append(eot_id)
print(f"EOS/EOT IDs: {eos_ids}")

# ============================================================
# 13. PROMPT BUILDERS
# ============================================================
def build_explicit_formatted_prompt(topic_title, t_cefr):
    sys_instr = (
        f"You are an expert English language teacher demonstrating CEFR proficiency levels. "
        f"Your task is to write a flawless, grammatically correct text responding to this prompt: '{topic_title}'. "
        f"The output must serve as a perfect textbook example of strictly {t_cefr} level English. "
        f"If the requested target level is A1/A2, use very simple vocabulary, short sentences, and primitive structures. "
        f"If the requested target level is C1/C2, utilize highly advanced vocabulary, idioms, and complex sentence patterns. "
        f"Write only the direct response. Do not write any meta-commentary, greetings, or conversational pleasantries."
    )
    messages = [{"role": "system", "content": sys_instr}, {"role": "user", "content": topic_title}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_blind_formatted_prompt(topic_title):
    sys_instr = (
        f"You are an expert English language teacher demonstrating CEFR proficiency levels. "
        f"Your task is to write a flawless, grammatically correct text responding to this prompt: '{topic_title}'. "
        f"If the requested target level is A1/A2, use very simple vocabulary, short sentences, and primitive structures. "
        f"If the requested target level is C1/C2, utilize highly advanced vocabulary, idioms, and complex sentence patterns. "
        f"Write only the direct response. Do not write any meta-commentary, greetings, or conversational pleasantries."
    )
    messages = [{"role": "system", "content": sys_instr}, {"role": "user", "content": topic_title}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ============================================================
# 14. STREAMER FOR EXACT TTFT USING HF GENERATE()
# ============================================================
class FirstTokenTimingStreamer(BaseStreamer):
    def __init__(self):
        self.prompt_received = False
        self.first_token_timestamp = None

    def put(self, value):
        if not self.prompt_received:
            self.prompt_received = True
            return
        if self.first_token_timestamp is None:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            self.first_token_timestamp = time.perf_counter()

    def end(self):
        pass

# ============================================================
# 15. GPU CLEANUP
# ============================================================
def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# ============================================================
# 16. COMMON PEFT BENCHMARK
# ============================================================
def benchmark_peft_model(method_name, adapter_repo_id, output_csv_path):
    print("\n" + "=" * 75 + f"\n🚀 BENCHMARKING: {method_name}\n" + "=" * 75)
    set_seed(42)
    cleanup_gpu()

    print("\nLoading frozen base model...")
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
    base_model.eval()
    base_model.config.use_cache = True

    print(f"Loading PEFT adapter:\n{adapter_repo_id}")
    model = PeftModel.from_pretrained(base_model, adapter_repo_id, is_trainable=False)
    model.eval()

    print("Running warm-up generation...")
    warmup_prompt = build_explicit_formatted_prompt("Warmup Prompt", "B1")
    warmup_inputs = tokenizer(warmup_prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        _ = model.generate(**warmup_inputs, max_new_tokens=10, do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_ids)
    torch.cuda.synchronize()

    latency_records = []
    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"{method_name} Latency"):
        t_cefr = str(row["cefr"]).strip().upper()
        formatted_prompt = build_explicit_formatted_prompt(row["topic_title"], t_cefr)
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
        prompt_length = inputs.input_ids.shape[1]
        streamer = FirstTokenTimingStreamer()

        torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_k=TOP_K,
                repetition_penalty=REPETITION_PENALTY, do_sample=True, use_cache=True,
                streamer=streamer, pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_ids
            )

        torch.cuda.synchronize()
        end_time = time.perf_counter()

        ttlt = end_time - start_time
        ttft = streamer.first_token_timestamp - start_time if streamer.first_token_timestamp is not None else ttlt

        generated_ids = output_ids[0, prompt_length:]
        num_tokens = int(generated_ids.numel())
        gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        gen_speed = num_tokens / ttlt if ttlt > 0 else 0.0
        decode_speed = (num_tokens - 1) / (ttlt - ttft) if (num_tokens > 1 and (ttlt - ttft) > 0) else gen_speed

        latency_records.append({
            "topic_id": row["topic_id"], "topic_title": row["topic_title"], "target_cefr": t_cefr,
            "llm_output": gen_text, "num_generated_tokens": num_tokens, "ttft_seconds": round(ttft, 4),
            "ttlt_seconds": round(ttlt, 4), "gen_speed_tokens_per_sec": round(gen_speed, 2),
            "decode_speed_tokens_per_sec": round(decode_speed, 2)
        })

    df_latency = pd.DataFrame(latency_records)
    df_latency.to_csv(output_csv_path, index=False)

    summary = {
        "method": method_name,
        "avg_ttft": df_latency["ttft_seconds"].mean(), "avg_ttlt": df_latency["ttlt_seconds"].mean(),
        "avg_gen_speed": df_latency["gen_speed_tokens_per_sec"].mean(),
        "avg_decode_speed": df_latency["decode_speed_tokens_per_sec"].mean(),
        "avg_generated_tokens": df_latency["num_generated_tokens"].mean(),
    }

    print(f"\n✔ Saved:\n{output_csv_path}\n\n{method_name}")
    print(f"TTFT         : {summary['avg_ttft']:.4f} s\nTTLT         : {summary['avg_ttlt']:.4f} s")
    print(f"Generation   : {summary['avg_gen_speed']:.2f} tok/s\nDecode Speed : {summary['avg_decode_speed']:.2f} tok/s")

    del model, base_model
    cleanup_gpu()
    return df_latency, summary

# ============================================================
# 17 & 18. BENCHMARK LoRA & STANDARD PREFIX-TUNING
# ============================================================
df_lora_latency, lora_summary = benchmark_peft_model("LoRA (6k)", LORA_REPO_ID, LORA_CSV_PATH)
df_standard_pt_latency, standard_pt_summary = benchmark_peft_model("Standard Prefix-Tuning (6k)", STANDARD_PT_REPO_ID, STANDARD_PT_CSV_PATH)

# ============================================================
# 19. CEFR MULTI-PREFIX TUNING ~537M CONTROLLER
# ============================================================
class CEFRMultiPrefixController(nn.Module):
    def __init__(self, config, num_virtual_tokens=30, num_classes=6, mlp_hidden_dim=7696):
        super().__init__()
        self.num_virtual_tokens = num_virtual_tokens
        self.num_classes = num_classes
        self.num_layers = config.num_hidden_layers
        self.hidden_size = config.hidden_size
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
        self.prefix_embeddings = nn.Embedding(num_classes * num_virtual_tokens, self.hidden_size)

        flat_out_dim = self.num_layers * 2 * self.num_kv_heads * self.head_dim
        self.prefix_mlp = nn.Sequential(
            nn.Linear(self.hidden_size, mlp_hidden_dim),
            nn.Tanh(),
            nn.Linear(mlp_hidden_dim, flat_out_dim),
        )

    def forward(self, cefr_ids):
        batch_size = cefr_ids.shape[0]
        class_offsets = (cefr_ids * self.num_virtual_tokens).unsqueeze(1)
        base_indices = torch.arange(self.num_virtual_tokens, device=cefr_ids.device).unsqueeze(0)
        token_indices = class_offsets + base_indices

        prefix_tokens = self.prefix_embeddings(token_indices)
        past_kv_flat = self.prefix_mlp(prefix_tokens)

        past_kv = past_kv_flat.view(batch_size, self.num_virtual_tokens, self.num_layers, 2, self.num_kv_heads, self.head_dim)
        past_kv = past_kv.permute(2, 3, 0, 4, 1, 5)

        cache = DynamicCache()
        for layer_idx in range(self.num_layers):
            cache.update(past_kv[layer_idx, 0], past_kv[layer_idx, 1], layer_idx=layer_idx)
        return cache

# ============================================================
# 20. COMMON MANUAL SAMPLING FUNCTION
# ============================================================
def sample_next_token(logits, prompt_token_ids, generated_tokens):
    logits = logits.clone()
    previous_ids = set(prompt_token_ids + generated_tokens)
    for tok_id in previous_ids:
        if logits[0, tok_id] > 0:
            logits[0, tok_id] /= REPETITION_PENALTY
        else:
            logits[0, tok_id] *= REPETITION_PENALTY

    logits = logits / TEMPERATURE
    values, indices = torch.topk(logits, TOP_K, dim=-1)
    filtered_logits = torch.full_like(logits, float("-inf"))
    filtered_logits.scatter_(1, indices, values)
    probs = F.softmax(filtered_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

# ============================================================
# 21 & 22. BENCHMARK CEFR MULTI-PREFIX TUNING ~537M
# ============================================================
def benchmark_cefr_pt_537m():
    print("\n" + "=" * 75 + "\n🚀 BENCHMARKING: CEFR PREFIX-TUNING (~537M)\n" + "=" * 75)
    set_seed(42)
    cleanup_gpu()

    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
    base_model.eval()
    base_model.config.use_cache = True

    prefix_controller = CEFRMultiPrefixController(base_model.config, num_virtual_tokens=30, num_classes=6, mlp_hidden_dim=7696).to(device, dtype=torch.bfloat16)
    weights_path = hf_hub_download(repo_id=CEFR_PT_537M_REPO_ID, filename="cefr_prefix_tuning_537m_best_weights.pt")
    prefix_controller.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    prefix_controller.eval()

    controller_params = sum(p.numel() for p in prefix_controller.parameters())
    print(f"Controller parameters: {controller_params:,}")
    assert controller_params == 536_698_384, "Unexpected CEFR PT 537M parameter count."

    print("Running CEFR PT warm-up...")
    warm_prompt = build_blind_formatted_prompt("Warmup Prompt")
    warm_inputs = tokenizer(warm_prompt, return_tensors="pt").to(device)
    warm_cefr = torch.tensor([label_map["B1"]], dtype=torch.long, device=device)

    with torch.inference_mode():
        warm_cache = prefix_controller(warm_cefr)
        n_prefix = prefix_controller.num_virtual_tokens
        prefix_mask = torch.ones(1, n_prefix, dtype=warm_inputs.attention_mask.dtype, device=device)
        full_mask = torch.cat([prefix_mask, warm_inputs.attention_mask], dim=1)

        position_ids = full_mask.long().cumsum(-1) - 1
        position_ids.masked_fill_(full_mask == 0, 1)
        position_ids = position_ids[:, n_prefix:]

        prompt_len = warm_inputs.input_ids.shape[1]
        cache_position = torch.arange(n_prefix, n_prefix + prompt_len, dtype=torch.long, device=device)

        _ = base_model(input_ids=warm_inputs.input_ids, attention_mask=full_mask, position_ids=position_ids,
                       cache_position=cache_position, past_key_values=warm_cache, use_cache=True)
    torch.cuda.synchronize()

    latency_records = []
    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="CEFR PT ~537M Latency"):
        t_cefr = str(row["cefr"]).strip().upper()
        formatted_prompt = build_blind_formatted_prompt(row["topic_title"])
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
        input_ids, attention_mask = inputs.input_ids, inputs.attention_mask
        prompt_length = input_ids.shape[1]
        prompt_token_ids = input_ids[0].tolist()
        cefr_tensor = torch.tensor([label_map[t_cefr]], dtype=torch.long, device=device)
        generated_tokens = []

        torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.inference_mode():
            past_key_values = prefix_controller(cefr_tensor)
            n_prefix = prefix_controller.num_virtual_tokens
            prefix_mask = torch.ones(1, n_prefix, dtype=attention_mask.dtype, device=device)
            current_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

            position_ids = current_attention_mask.long().cumsum(-1) - 1
            position_ids.masked_fill_(current_attention_mask == 0, 1)
            position_ids = position_ids[:, n_prefix:]
            cache_position = torch.arange(n_prefix, n_prefix + prompt_length, dtype=torch.long, device=device)

            outputs = base_model(input_ids=input_ids, attention_mask=current_attention_mask, position_ids=position_ids,
                                 cache_position=cache_position, past_key_values=past_key_values, use_cache=True)
            past_key_values = outputs.past_key_values
            logits = outputs.logits[:, -1, :]
            next_token_id = sample_next_token(logits, prompt_token_ids, generated_tokens)

            torch.cuda.synchronize()
            ttft = time.perf_counter() - start_time
            tok_item = next_token_id.item()
            generated_tokens.append(tok_item)
            current_cache_length = n_prefix + prompt_length

            if tok_item not in eos_ids:
                for _ in range(1, MAX_NEW_TOKENS):
                    current_attention_mask = torch.cat([current_attention_mask, torch.ones(1, 1, dtype=current_attention_mask.dtype, device=device)], dim=1)
                    next_position_id = torch.tensor([[current_cache_length]], dtype=torch.long, device=device)
                    next_cache_position = torch.tensor([current_cache_length], dtype=torch.long, device=device)

                    outputs = base_model(input_ids=next_token_id, attention_mask=current_attention_mask, position_ids=next_position_id,
                                         cache_position=next_cache_position, past_key_values=past_key_values, use_cache=True)
                    past_key_values = outputs.past_key_values
                    logits = outputs.logits[:, -1, :]
                    next_token_id = sample_next_token(logits, prompt_token_ids, generated_tokens)

                    tok_item = next_token_id.item()
                    generated_tokens.append(tok_item)
                    current_cache_length += 1
                    if tok_item in eos_ids:
                        break

        torch.cuda.synchronize()
        ttlt = time.perf_counter() - start_time
        num_tokens = len(generated_tokens)
        gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        gen_speed = num_tokens / ttlt if ttlt > 0 else 0.0
        decode_speed = (num_tokens - 1) / (ttlt - ttft) if (num_tokens > 1 and (ttlt - ttft) > 0) else gen_speed

        latency_records.append({
            "topic_id": row["topic_id"], "topic_title": row["topic_title"], "target_cefr": t_cefr,
            "llm_output": gen_text, "num_generated_tokens": num_tokens, "ttft_seconds": round(ttft, 4),
            "ttlt_seconds": round(ttlt, 4), "gen_speed_tokens_per_sec": round(gen_speed, 2),
            "decode_speed_tokens_per_sec": round(decode_speed, 2),
        })

    df_latency = pd.DataFrame(latency_records)
    df_latency.to_csv(CEFR_PT_537M_CSV_PATH, index=False)

    summary = {
        "method": "CEFR Prefix-Tuning (~537M)",
        "avg_ttft": df_latency["ttft_seconds"].mean(), "avg_ttlt": df_latency["ttlt_seconds"].mean(),
        "avg_gen_speed": df_latency["gen_speed_tokens_per_sec"].mean(),
        "avg_decode_speed": df_latency["decode_speed_tokens_per_sec"].mean(),
        "avg_generated_tokens": df_latency["num_generated_tokens"].mean(),
    }

    print(f"\n✔ Saved:\n{CEFR_PT_537M_CSV_PATH}\n\nCEFR Prefix-Tuning (~537M)")
    print(f"TTFT         : {summary['avg_ttft']:.4f} s\nTTLT         : {summary['avg_ttlt']:.4f} s")
    print(f"Generation   : {summary['avg_gen_speed']:.2f} tok/s\nDecode Speed : {summary['avg_decode_speed']:.2f} tok/s")

    del prefix_controller, base_model
    cleanup_gpu()
    return df_latency, summary

df_cefr_pt_537m_latency, cefr_pt_537m_summary = benchmark_cefr_pt_537m()

# ============================================================
# 23 - 27. PROPOSED UNCOMPRESSED CEFR-GATED PMT
# ============================================================
class PureCEFRGatedPMTController(nn.Module):
    def __init__(self, num_layers=32, hidden_dim=4096, num_classes=6):
        super().__init__()
        self.num_layers, self.hidden_dim = num_layers, hidden_dim
        self.alpha = nn.Parameter(torch.tensor(0.1))
        self.cefr_embedding = nn.Embedding(num_classes, hidden_dim)
        self.memory = nn.Parameter(torch.empty(num_layers, hidden_dim, hidden_dim))
        nn.init.zeros_(self.memory)

def locate_controller_checkpoint(repo_id):
    api = HfApi()
    repo_files = api.list_repo_files(repo_id=repo_id)
    print("\nRepository files:"); [print(f"  - {f}") for f in repo_files]
    candidate_files = [f for f in repo_files if f.endswith((".pt", ".pth", ".bin"))]
    controller_candidates = [f for f in candidate_files if any(x in f.lower() for x in ["controller", "prefixmemory", "pmt", "weight"])]

    if len(controller_candidates) == 1: selected_file = controller_candidates[0]
    elif len(candidate_files) == 1: selected_file = candidate_files[0]
    else: raise RuntimeError(f"\nCould not uniquely determine PMT controller checkpoint.\nCandidate files:\n{chr(10).join(candidate_files)}")
    print(f"\nSelected PMT controller file:\n{selected_file}")
    return hf_hub_download(repo_id=repo_id, filename=selected_file)

def extract_pmt_weights(checkpoint):
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint and isinstance(checkpoint["state_dict"], dict): checkpoint = checkpoint["state_dict"]
    elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint and isinstance(checkpoint["model_state_dict"], dict): checkpoint = checkpoint["model_state_dict"]
    if not isinstance(checkpoint, dict): raise RuntimeError("Unexpected PMT checkpoint format.")

    print("\nPMT checkpoint tensor keys:")
    tensors = [(k, v) for k, v in checkpoint.items() if torch.is_tensor(v)]
    for key, tensor in tensors: print(f"  {key:<60} {tuple(tensor.shape)}")

    memory_candidates = [(k, t) for k, t in tensors if tuple(t.shape) == (32, 4096, 4096)]
    embedding_candidates = [(k, t) for k, t in tensors if tuple(t.shape) == (6, 4096)]
    alpha_candidates = [(k, t) for k, t in tensors if t.numel() == 1]

    if len(memory_candidates) != 1: raise RuntimeError(f"Expected exactly one [32,4096,4096] PMT memory tensor, found {len(memory_candidates)}.")
    if len(embedding_candidates) != 1: raise RuntimeError(f"Expected exactly one [6,4096] CEFR embedding tensor, found {len(embedding_candidates)}.")
    if len(alpha_candidates) < 1: raise RuntimeError("Could not identify scalar PMT alpha.")

    named_alpha = [(k, t) for k, t in alpha_candidates if "alpha" in k.lower()]
    if len(named_alpha) == 1: alpha_key, alpha_tensor = named_alpha[0]
    elif len(alpha_candidates) == 1: alpha_key, alpha_tensor = alpha_candidates[0]
    else: raise RuntimeError("Multiple scalar tensors found and alpha could not be uniquely identified.")

    memory_key, memory_tensor = memory_candidates[0]
    embedding_key, embedding_tensor = embedding_candidates[0]
    print(f"\nMapped proposed PMT tensors:\nMemory    : {memory_key}\nEmbedding : {embedding_key}\nAlpha     : {alpha_key}")
    return memory_tensor, embedding_tensor, alpha_tensor

def benchmark_cefr_gated_pmt():
    print("\n" + "=" * 75 + "\n🚀 BENCHMARKING: PROPOSED CEFR-GATED PMT (~537M)\n" + "=" * 75)
    set_seed(42)
    cleanup_gpu()

    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
    base_model.eval()
    base_model.config.use_cache = True

    pmt_controller = PureCEFRGatedPMTController().to(device, dtype=torch.bfloat16)
    controller_params = sum(p.numel() for p in pmt_controller.parameters())
    print(f"Expected PMT controller parameters: {controller_params:,}")
    assert controller_params == 536_895_489, "Unexpected proposed PMT parameter count."

    pmt_weights_path = locate_controller_checkpoint(CEFR_GATED_PMT_REPO_ID)
    checkpoint = torch.load(pmt_weights_path, map_location="cpu", weights_only=True)
    memory_tensor, embedding_tensor, alpha_tensor = extract_pmt_weights(checkpoint)

    with torch.no_grad():
        pmt_controller.memory.copy_(memory_tensor.to(device=device, dtype=torch.bfloat16))
        pmt_controller.cefr_embedding.weight.copy_(embedding_tensor.to(device=device, dtype=torch.bfloat16))
        pmt_controller.alpha.copy_(alpha_tensor.to(device=device, dtype=torch.bfloat16).reshape(()))

    del checkpoint, memory_tensor, embedding_tensor, alpha_tensor
    gc.collect()
    pmt_controller.eval()
    print(f"Loaded alpha: {pmt_controller.alpha.item():.6f}")

    active_cefr_embedding = None
    hook_handles = []

    def make_pmt_hook(layer_idx):
        def hook_fn(module, args, kwargs, output):
            nonlocal active_cefr_embedding
            if active_cefr_embedding is None: return output

            hidden_states = args[0] if len(args) > 0 else kwargs.get("hidden_states")
            if hidden_states is None: return output
            attn_output = output[0] if isinstance(output, tuple) else output

            phi_x = F.elu(hidden_states)
            gated_x = phi_x * active_cefr_embedding
            memory_bias = torch.matmul(gated_x, pmt_controller.memory[layer_idx])
            final_attn_output = attn_output + (pmt_controller.alpha * memory_bias)

            return (final_attn_output,) + output[1:] if isinstance(output, tuple) else final_attn_output
        return hook_fn

    for layer_idx in range(32):
        handle = base_model.model.layers[layer_idx].self_attn.register_forward_hook(make_pmt_hook(layer_idx), with_kwargs=True)
        hook_handles.append(handle)

    print("Running proposed PMT warm-up...")
    warm_prompt = build_blind_formatted_prompt("Warmup Prompt")
    warm_inputs = tokenizer(warm_prompt, return_tensors="pt").to(device)
    warm_cefr_tensor = torch.tensor([label_map["B1"]], dtype=torch.long, device=device)

    with torch.inference_mode():
        active_cefr_embedding = pmt_controller.cefr_embedding(warm_cefr_tensor).unsqueeze(1)
        _ = base_model(input_ids=warm_inputs.input_ids, attention_mask=warm_inputs.attention_mask, use_cache=True)
    torch.cuda.synchronize()

    latency_records = []
    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="CEFR-Gated PMT Latency"):
        t_cefr = str(row["cefr"]).strip().upper()
        formatted_prompt = build_blind_formatted_prompt(row["topic_title"])
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
        input_ids, attention_mask = inputs.input_ids, inputs.attention_mask
        prompt_length = input_ids.shape[1]
        prompt_token_ids = input_ids[0].tolist()
        cefr_tensor = torch.tensor([label_map[t_cefr]], dtype=torch.long, device=device)

        generated_tokens = []
        past_key_values = None
        current_attention_mask = attention_mask

        torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.inference_mode():
            active_cefr_embedding = pmt_controller.cefr_embedding(cefr_tensor).unsqueeze(1)

            for step in range(MAX_NEW_TOKENS):
                model_input = input_ids if step == 0 else next_token_id

                if step > 0:
                    current_attention_mask = torch.cat([current_attention_mask, torch.ones(1, 1, dtype=current_attention_mask.dtype, device=device)], dim=-1)

                outputs = base_model(
                    input_ids=model_input, attention_mask=current_attention_mask,
                    past_key_values=past_key_values, use_cache=True
                )
                past_key_values = outputs.past_key_values
                logits = outputs.logits[:, -1, :]

                next_token_id = sample_next_token(logits, prompt_token_ids, generated_tokens)

                if step == 0:
                    torch.cuda.synchronize()
                    ttft = time.perf_counter() - start_time

                tok_item = next_token_id.item()
                generated_tokens.append(tok_item)
                if tok_item in eos_ids: break

        torch.cuda.synchronize()
        ttlt = time.perf_counter() - start_time
        num_tokens = len(generated_tokens)
        gen_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        gen_speed = num_tokens / ttlt if ttlt > 0 else 0.0
        decode_speed = (num_tokens - 1) / (ttlt - ttft) if (num_tokens > 1 and (ttlt - ttft) > 0) else gen_speed

        latency_records.append({
            "topic_id": row["topic_id"], "topic_title": row["topic_title"], "target_cefr": t_cefr,
            "llm_output": gen_text, "num_generated_tokens": num_tokens, "ttft_seconds": round(ttft, 4),
            "ttlt_seconds": round(ttlt, 4), "gen_speed_tokens_per_sec": round(gen_speed, 2),
            "decode_speed_tokens_per_sec": round(decode_speed, 2),
        })

    df_latency = pd.DataFrame(latency_records)
    df_latency.to_csv(CEFR_GATED_PMT_CSV_PATH, index=False)

    summary = {
        "method": "CEFR-Gated PMT (Proposed)",
        "avg_ttft": df_latency["ttft_seconds"].mean(), "avg_ttlt": df_latency["ttlt_seconds"].mean(),
        "avg_gen_speed": df_latency["gen_speed_tokens_per_sec"].mean(),
        "avg_decode_speed": df_latency["decode_speed_tokens_per_sec"].mean(),
        "avg_generated_tokens": df_latency["num_generated_tokens"].mean(),
    }

    print(f"\n✔ Saved:\n{CEFR_GATED_PMT_CSV_PATH}\n\nCEFR-Gated PMT (Proposed)")
    print(f"TTFT         : {summary['avg_ttft']:.4f} s\nTTLT         : {summary['avg_ttlt']:.4f} s")
    print(f"Generation   : {summary['avg_gen_speed']:.2f} tok/s\nDecode Speed : {summary['avg_decode_speed']:.2f} tok/s")

    for handle in hook_handles: handle.remove()
    del pmt_controller, base_model
    cleanup_gpu()
    return df_latency, summary

df_cefr_gated_pmt_latency, cefr_gated_pmt_summary = benchmark_cefr_gated_pmt()

# ============================================================
# 28. COMPILE FOUR-MODEL SUMMARY & 29. SAVE
# ============================================================
all_summaries = [lora_summary, standard_pt_summary, cefr_pt_537m_summary, cefr_gated_pmt_summary]
hardware_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

summary_text = "=" * 110 + "\n THESIS INFERENCE LATENCY BENCHMARK — FOUR ADDITIONAL REPRESENTATIVE MODELS\n" + "=" * 110 + "\n"
summary_text += f"Benchmark Location : {LATENCY_DIR}\nBenchmark Dataset  : {SUBSET_12_CSV}\nHardware Engine    : {hardware_name}\nEvaluation Mode    : Single Prompt Generation (Batch Size = 1)\n"
summary_text += f"Prompts            : {len(df_subset)}\nMax New Tokens     : {MAX_NEW_TOKENS}\nTemperature        : {TEMPERATURE}\nTop-K              : {TOP_K}\nRepetition Penalty : {REPETITION_PENALTY}\n"
summary_text += "=" * 110 + "\n\n"
summary_text += f"{'Method':<38} | {'Avg TTFT':<12} | {'Avg TTLT':<12} | {'Gen Speed':<14} | {'Decode Speed':<14}\n" + "-" * 110 + "\n"

for res in all_summaries:
    summary_text += f"{res['method']:<38} | {res['avg_ttft']:<12.4f} | {res['avg_ttlt']:<12.4f} | {res['avg_gen_speed']:<14.2f} | {res['avg_decode_speed']:<14.2f}\n"

summary_text += "-" * 110 + "\n\nDETAILED BREAKDOWN\n" + "-" * 110 + "\n"

for res in all_summaries:
    summary_text += f"{res['method']}\n  TTFT         : {res['avg_ttft']:.4f} s\n  TTLT         : {res['avg_ttlt']:.4f} s\n  Gen Speed    : {res['avg_gen_speed']:.2f} tokens/s\n  Decode Speed : {res['avg_decode_speed']:.2f} tokens/s\n  Avg Tokens   : {res['avg_generated_tokens']:.2f}\n\n"

with open(SUMMARY_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\n" + summary_text)
print("\n" + "=" * 70 + "\n🎉 FOUR-MODEL LATENCY BENCHMARK COMPLETE\n" + "=" * 70)
print(f"\nArtifacts saved:\n{LORA_CSV_PATH}\n{STANDARD_PT_CSV_PATH}\n{CEFR_PT_537M_CSV_PATH}\n{CEFR_GATED_PMT_CSV_PATH}\n{SUMMARY_TXT_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 130.8 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Mounted at /content/drive
✔ Hugging Face authentication successful.
Using execution device: cuda
GPU: NVIDIA L4

12-PROMPT LATENCY BENCHMARK
Dataset: /content/drive/MyDrive/Mohammd_Thesis/Results/Latency_Comparison/latency_benchmark_subset_12.csv
Rows: 12
cefr
A1    2
A2    2
B1    2
B2    2
C1    2
C2    2
Name: count, dtype: int64


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

...

pure_pmt_controller_weights.pt: reconstructing file:   0%|          |  0.00B / 1.07GB            

pure_pmt_controller_weights.pt: downloading bytes:           |  0.00B            


PMT checkpoint tensor keys:
  alpha                                                        ()
  M                                                            (32, 4096, 4096)
  cefr_embeddings.weight                                       (6, 4096)

Mapped proposed PMT tensors:
Memory    : M
Embedding : cefr_embeddings.weight
Alpha     : alpha
Loaded alpha: 0.100098
Running proposed PMT warm-up...


CEFR-Gated PMT Latency: 100%|██████████| 12/12 [02:07<00:00, 10.61s/it]



✔ Saved:
/content/drive/MyDrive/Mohammd_Thesis/Results/Latency_Comparison/cefr_gated_pmt_latency.csv

CEFR-Gated PMT (Proposed)
TTFT         : 0.1018 s
TTLT         : 10.6089 s
Generation   : 13.16 tok/s
Decode Speed : 13.20 tok/s

 THESIS INFERENCE LATENCY BENCHMARK — FOUR ADDITIONAL REPRESENTATIVE MODELS
Benchmark Location : /content/drive/MyDrive/Mohammd_Thesis/Results/Latency_Comparison
Benchmark Dataset  : /content/drive/MyDrive/Mohammd_Thesis/Results/Latency_Comparison/latency_benchmark_subset_12.csv
Hardware Engine    : NVIDIA L4
Evaluation Mode    : Single Prompt Generation (Batch Size = 1)
Prompts            : 12
Max New Tokens     : 200
Temperature        : 0.6
Top-K              : 50
Repetition Penalty : 1.15

Method                                 | Avg TTFT     | Avg TTLT     | Gen Speed      | Decode Speed  
--------------------------------------------------------------------------------------------------------------
LoRA (6k)                              | 0.1327       